In [ ]:
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score

In [ ]:
GOOGLE_DRIVE_DATA_DIR = ''

OUTPUT_DIRNAME = 't4_hybrid_outputs'

# использование рыночных вороятностей, где они есть
USE_MARKET_WHEN_AVAILABLE = True
MARKET_BLEND_WEIGHT = 1.0

# настройки
PROMPT_CHAR_LIMIT = 2500
QUESTION_MAX_FEATURES = 50_000
PROMPT_MAX_FEATURES = 120_000
LOGREG_C = 2.0
LOGREG_MAX_ITER = 500
MIN_CALIB_ROWS = 2000
CALIB_FRACTION = 0.18

TRAIN_FILENAME = 'final_train_data.csv'
VALID_FILENAME = 'final_valid_data.csv'
TEST_FILENAME = 'final_test_data.csv'

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', repr(exc))


In [ ]:
# анализ данных и препроцессинг
LEAK_RE = re.compile(
    r"[ \\t]*Please note that for the YES event, "
    r"Polymarket participants assign a probability of[^\\n]*\\n?",
    re.I,
)

def unique_paths(paths):
    seen = set()
    out = []
    for p in paths:
        if not p:
            continue
        path = Path(p).expanduser()
        key = str(path)
        if key not in seen:
            out.append(path)
            seen.add(key)
    return out

def has_split_files(folder: Optional[Path]) -> bool:
    return bool(
        folder
        and folder.exists()
        and (folder / TRAIN_FILENAME).exists()
        and (folder / VALID_FILENAME).exists()
        and (folder / TEST_FILENAME).exists()
    )

def discover_data_dir() -> Path:
    drive_root = Path('/content/drive')
    my_drive = drive_root / 'MyDrive'

    candidates = unique_paths([
        GOOGLE_DRIVE_DATA_DIR,
        my_drive / 'final_data',
        my_drive / 'data',
        my_drive / 'Colab Notebooks' / 'final_data',
        Path('/content'),
        Path.cwd(),
        Path.cwd() / 'final_data',
    ])

    for cand in candidates:
        if has_split_files(cand):
            return cand

    search_roots = [p for p in [my_drive, Path('/content'), Path.cwd()] if p.exists()]
    for root in search_roots:
        try:
            for train_path in root.rglob(TRAIN_FILENAME):
                folder = train_path.parent
                if has_split_files(folder):
                    return folder
        except Exception as exc:
            print('Search failed for', root, '->', repr(exc))

    raise FileNotFoundError(
        'Could not find a folder with final_train_data.csv, final_valid_data.csv, final_test_data.csv. '
        'Set GOOGLE_DRIVE_DATA_DIR explicitly.'
    )

def clean_prompt(prompt: str) -> str:
    return LEAK_RE.sub('', str(prompt or ''))

def prepare_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['question_close_date'], errors='coerce')
    out['question_text'] = out['question_text'].fillna('')
    out['prompt_clean'] = (
        out['prompt'].fillna('').astype(str).map(clean_prompt).str.slice(0, PROMPT_CHAR_LIMIT)
    )
    out['log_volume'] = np.log1p(pd.to_numeric(out['volume'], errors='coerce').fillna(0.0))
    out['meta_text'] = (
        dt.dt.year.fillna(0).astype(int).astype(str)
        + ' [MONTH] ' + dt.dt.month.fillna(0).astype(int).astype(str)
        + ' [DOW] ' + dt.dt.dayofweek.fillna(0).astype(int).astype(str)
    )
    return out

def brier_or_nan(y_true: np.ndarray, p: np.ndarray) -> float:
    mask = ~np.isnan(p)
    if mask.sum() == 0:
        return float('nan')
    return float(brier_score_loss(y_true[mask], p[mask]))

In [ ]:
# загрузка данных
DATA_DIR = discover_data_dir()
OUTPUT_DIR = DATA_DIR / OUTPUT_DIRNAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

train_df = pd.read_csv(DATA_DIR / TRAIN_FILENAME)
valid_df = pd.read_csv(DATA_DIR / VALID_FILENAME)
test_df = pd.read_csv(DATA_DIR / TEST_FILENAME)

train_df = prepare_frame(train_df)
valid_df = prepare_frame(valid_df)
test_df = prepare_frame(test_df)

print('train:', train_df.shape)
print('valid:', valid_df.shape)
print('test :', test_df.shape)


In [ ]:
@dataclass #умная штучка для сокращения записи служебных слов
class HybridForecastModel:
    question_vectorizer: TfidfVectorizer
    prompt_vectorizer: TfidfVectorizer
    text_model: LogisticRegression
    text_calibrator: IsotonicRegression
    market_calibrator: Optional[IsotonicRegression]

    def transform_text(self, df: pd.DataFrame):
        return hstack([
            self.question_vectorizer.transform(df['question_text'] + ' [META] ' + df['meta_text']),
            self.prompt_vectorizer.transform(df['prompt_clean']),
            csr_matrix(df[['log_volume']].values),
        ])

    def predict_text(self, df: pd.DataFrame) -> np.ndarray:
        raw = self.text_model.predict_proba(self.transform_text(df))[:, 1]
        calibrated = self.text_calibrator.transform(raw)
        return np.clip(calibrated, 1e-4, 1 - 1e-4)

    def predict_market(self, df: pd.DataFrame) -> np.ndarray:
        raw_market = pd.to_numeric(df['prediction_polymarket'], errors='coerce').to_numpy(dtype=float)
        out = np.full(len(df), np.nan, dtype=float)
        if self.market_calibrator is None:
            return out
        mask = ~np.isnan(raw_market)
        if mask.any():
            out[mask] = self.market_calibrator.transform(raw_market[mask])
        return np.clip(out, 1e-4, 1 - 1e-4)

    def predict_hybrid(self, df: pd.DataFrame) -> pd.DataFrame:
        text_pred = self.predict_text(df)
        market_pred = self.predict_market(df)
        final_pred = text_pred.copy()
        source = np.array(['text_model'] * len(df), dtype=object)

        if USE_MARKET_WHEN_AVAILABLE and self.market_calibrator is not None:
            market_mask = ~np.isnan(market_pred)
            if MARKET_BLEND_WEIGHT >= 1.0:
                final_pred[market_mask] = market_pred[market_mask]
            else:
                final_pred[market_mask] = (
                    MARKET_BLEND_WEIGHT * market_pred[market_mask]
                    + (1.0 - MARKET_BLEND_WEIGHT) * text_pred[market_mask]
                )
            source[market_mask] = 'market_or_hybrid'

        return pd.DataFrame({
            'prediction_text_model': text_pred,
            'prediction_market_calibrated': market_pred,
            'prediction_model': np.clip(final_pred, 1e-4, 1 - 1e-4),
            'prediction_source': source,
        })

print('Model class ready')


In [ ]:
# обучение гибридной модели
def build_hybrid_model(train_df: pd.DataFrame) -> Tuple[HybridForecastModel, Dict[str, float]]:
    calib_rows = max(MIN_CALIB_ROWS, int(len(train_df) * CALIB_FRACTION))
    calib_rows = min(calib_rows, len(train_df) // 2)
    split_idx = len(train_df) - calib_rows

    fit_df = train_df.iloc[:split_idx].copy()
    calib_df = train_df.iloc[split_idx:].copy()

    y_fit = fit_df['resolution'].astype(int).to_numpy()
    y_calib = calib_df['resolution'].astype(int).to_numpy()

    question_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        sublinear_tf=True,
        max_features=QUESTION_MAX_FEATURES,
    )
    prompt_vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
        sublinear_tf=True,
        max_features=PROMPT_MAX_FEATURES,
        stop_words='english',
    )

    question_vectorizer.fit(fit_df['question_text'] + ' [META] ' + fit_df['meta_text'])
    prompt_vectorizer.fit(fit_df['prompt_clean'])

    def transform_local(df: pd.DataFrame):
        return hstack([
            question_vectorizer.transform(df['question_text'] + ' [META] ' + df['meta_text']),
            prompt_vectorizer.transform(df['prompt_clean']),
            csr_matrix(df[['log_volume']].values),
        ])

    x_fit = transform_local(fit_df)
    x_calib = transform_local(calib_df)

    text_model = LogisticRegression(
        C=LOGREG_C,
        max_iter=LOGREG_MAX_ITER,
        solver='liblinear',
        class_weight='balanced',
    )
    text_model.fit(x_fit, y_fit)

    text_calib_raw = text_model.predict_proba(x_calib)[:, 1]
    text_calibrator = IsotonicRegression(out_of_bounds='clip')
    text_calibrator.fit(text_calib_raw, y_calib)

    market_calibrator = None
    fit_market = pd.to_numeric(fit_df['prediction_polymarket'], errors='coerce').to_numpy(dtype=float)
    market_mask = ~np.isnan(fit_market)
    if market_mask.sum() >= 200:
        market_calibrator = IsotonicRegression(out_of_bounds='clip')
        market_calibrator.fit(fit_market[market_mask], y_fit[market_mask])

    hybrid = HybridForecastModel(
        question_vectorizer=question_vectorizer,
        prompt_vectorizer=prompt_vectorizer,
        text_model=text_model,
        text_calibrator=text_calibrator,
        market_calibrator=market_calibrator,
    )

    calib_preds = hybrid.predict_hybrid(calib_df)
    metrics = {
        'train_rows': int(len(train_df)),
        'fit_rows': int(len(fit_df)),
        'calib_rows': int(len(calib_df)),
        'calib_positive_rate': float(y_calib.mean()),
        'calib_brier_text_only': brier_or_nan(y_calib, calib_preds['prediction_text_model'].to_numpy()),
        'calib_brier_market_calibrated': brier_or_nan(
            y_calib, calib_preds['prediction_market_calibrated'].to_numpy()
        ),
        'calib_brier_hybrid': brier_or_nan(y_calib, calib_preds['prediction_model'].to_numpy()),
        'calib_auc_text_only': float(
            roc_auc_score(y_calib, calib_preds['prediction_text_model'].to_numpy())
        ),
    }
    return hybrid, metrics

model, train_metrics = build_hybrid_model(train_df)
print(json.dumps(train_metrics, ensure_ascii=False, indent=2, default=float))


In [ ]:
valid_preds = model.predict_hybrid(valid_df)
valid_eval = {
    'rows': int(len(valid_df)),
    'brier_model': brier_or_nan(valid_df['resolution'].astype(int).to_numpy(), valid_preds['prediction_model'].to_numpy(dtype=float)),
    'brier_text_only': brier_or_nan(valid_df['resolution'].astype(int).to_numpy(), valid_preds['prediction_text_model'].to_numpy(dtype=float)),
    'brier_market_calibrated': brier_or_nan(valid_df['resolution'].astype(int).to_numpy(), valid_preds['prediction_market_calibrated'].to_numpy(dtype=float)),
    'used_market_rows': int((valid_preds['prediction_source'] == 'market_or_hybrid').sum()),
    'used_text_rows': int((valid_preds['prediction_source'] == 'text_model').sum()),
}
print(json.dumps(valid_eval, ensure_ascii=False, indent=2, default=float))
display(valid_preds.head(10))


In [ ]:
# сохранение всех предсказаний
def save_split_predictions(
    model: HybridForecastModel,
    df: pd.DataFrame,
    split_name: str,
    out_dir: Path,
) -> Dict[str, float]:
    preds = model.predict_hybrid(df)
    out = df.copy()
    for col in preds.columns:
        out[col] = preds[col].to_numpy()

    out_path = out_dir / f'{split_name}_with_prediction_model.csv'
    out.to_csv(out_path, index=False, encoding='utf-8-sig')

    y = out['resolution'].astype(int).to_numpy()
    metrics = {
        'split': split_name,
        'rows': int(len(out)),
        'positive_rate': float(np.mean(y)),
        'brier_model': brier_or_nan(y, out['prediction_model'].to_numpy(dtype=float)),
        'brier_text_only': brier_or_nan(y, out['prediction_text_model'].to_numpy(dtype=float)),
        'brier_market_calibrated': brier_or_nan(y, out['prediction_market_calibrated'].to_numpy(dtype=float)),
        'brier_raw_polymarket': brier_or_nan(
            y, pd.to_numeric(out['prediction_polymarket'], errors='coerce').to_numpy(dtype=float)
        ),
        'used_market_rows': int((out['prediction_source'] == 'market_or_hybrid').sum()),
        'used_text_rows': int((out['prediction_source'] == 'text_model').sum()),
        'path': str(out_path),
    }

    metrics_path = out_dir / f'{split_name}_metrics.json'
    metrics_path.write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2, default=float),
        encoding='utf-8',
    )
    print(json.dumps(metrics, ensure_ascii=False, indent=2, default=float))
    return metrics

metrics_rows = []
for split_name, split_df in [
    ('final_train_data', train_df),
    ('final_valid_data', valid_df),
    ('final_test_data', test_df),
]:
    metrics_rows.append(save_split_predictions(model, split_df, split_name, OUTPUT_DIR))

summary_df = pd.DataFrame(metrics_rows)
summary_path = OUTPUT_DIR / 'metrics_summary.csv'
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

print('Saved summary to:', summary_path)
display(summary_df)


In [ ]:
(OUTPUT_DIR / 'train_calibration_metrics.json').write_text(
    json.dumps(train_metrics, ensure_ascii=False, indent=2, default=float),
    encoding='utf-8',
)
print('Saved train calibration metrics')


In [ ]:
for path in [
    OUTPUT_DIR / 'train_calibration_metrics.json',
    OUTPUT_DIR / 'final_train_data_with_prediction_model.csv',
    OUTPUT_DIR / 'final_valid_data_with_prediction_model.csv',
    OUTPUT_DIR / 'final_test_data_with_prediction_model.csv',
    OUTPUT_DIR / 'metrics_summary.csv',
]:
    print('OK' if path.exists() else 'MISS', path)
